# Decider 2B server on Colab

Runtime: **GPU** (Runtime > Change runtime type > T4).
Run top to bottom. Cell 4 prints a public URL — paste it back.

In [1]:
# Cell 1: install + GPU check
!pip install -q strands-decider
!python -c "import strands_decider; print('strands-decider', strands_decider.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 5.4 MB/s eta 0:00:00
strands-decider 0.1.0
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
! pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
# Cell 2: start server (subprocess, logs to /content/server.log), wait for /health.
# First run downloads ~4GB of weights — up to 15 min.
import json, subprocess, time, urllib.request

proc = subprocess.Popen(
    ['strands-decider', 'serve',
     'StrandsAgents/strands-decider-2B-qwen3.5-v1-2610',
     '--device', 'cuda', '--port', '8099'],
    stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
print('server pid', proc.pid)

for i in range(180):
    if proc.poll() is not None:
        print('SERVER DIED')
        print(open('/content/server.log').read())
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8099/health', timeout=2) as r:
            print('health:', json.loads(r.read()))
            break
    except Exception:
        time.sleep(5)
else:
    print('TIMEOUT')
    print(open('/content/server.log').read()[-3000:])

server pid 5272
health: {'status': 'ok', 'model': 'strands-decider-2B-qwen3.5-v1-2610', 'checkpoint': 'StrandsAgents/strands-decider-2B-qwen3.5-v1-2610', 'base_model': 'Qwen/Qwen3.5-2B-Base', 'num_slots': 24, 'max_length': 4096, 'temperature': 1.0545210242263665, 'device': 'cuda', 'prefix_cache': True, 'vision': False}


In [8]:
# Cell 3: smoke test — the 4 patch-ranking questions in ONE request (state encoded once).
# Prints raw JSON to pin exact answer field names.
import json, time, urllib.request

state = '''Buggy function:
def clamp(x, lo, hi):
    return min(max(x, hi), lo)
Failing test output:
assert clamp(5, 0, 10) == 5  # got 0
Candidate patch:
def clamp(x, lo, hi):
    return min(max(x, lo), hi)'''

questions = {
  'addresses_failure': {'type': 'noul',
    'instructions': 'Does this patch change the code path exercised by the failing assertion?',
    'criteria': {'true': 'the patch modifies the expression or branch the assertion checks',
                 'false': 'the patch leaves the failing code path unchanged'}},
  'side_effects': {'type': 'noul',
    'instructions': 'Does the patch change behavior unrelated to the failing test?',
    'criteria': {'true': 'behavior outside the failing assertion changes',
                 'false': 'only the failing behavior is affected'}},
  'likely_passes': {'type': 'score',
    'instructions': 'How likely is this patch to make the failing test pass?',
    'criteria': ['very unlikely', 'unlikely', 'likely', 'very likely']},
}

body = json.dumps({'state': state, 'model': 'hobson-latest', 'questions': questions}).encode()
req = urllib.request.Request('http://127.0.0.1:8099/v1/systemone', data=body,
                             headers={'content-type': 'application/json'})
t0 = time.time()
with urllib.request.urlopen(req) as r:
    payload = json.loads(r.read())
print(f'round trip: {(time.time()-t0)*1000:.0f}ms')
print(json.dumps(payload, indent=2))

round trip: 3641ms
{
  "model": "strands-decider-2B-qwen3.5-v1-2610",
  "answers": {
    "addresses_failure": {
      "type": "noul",
      "noul": 0.6049
    },
    "side_effects": {
      "type": "noul",
      "noul": 0.3831
    },
    "likely_passes": {
      "type": "score",
      "score": 1.485,
      "legend": {
        "0": "very unlikely",
        "1": "unlikely",
        "2": "likely",
        "3": "very likely"
      },
      "probabilities": {
        "0": 0.2434,
        "1": 0.2659,
        "2": 0.2529,
        "3": 0.2378
      },
      "confidence": 0.3371
    }
  },
  "usage": {
    "input_tokens": 308,
    "output_tokens": 3
  },
  "latency_ms": 3630.73
}


In [9]:
# Cell 4: tunnel via cloudflared (no account). Prints PUBLIC URL — paste it back.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess, time, re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8099'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    time.sleep(1)
print('PUBLIC URL:', url)

PUBLIC URL: https://module-prev-revealed-disciplinary.trycloudflare.com


In [10]:
# Cell 5: verify tunnel end-to-end, then keep runtime alive. Interrupt to stop.
import json, time, urllib.request
with urllib.request.urlopen(url + '/health', timeout=30) as r:
    print('via tunnel:', json.loads(r.read()))
print('serving. keep this cell running.')
while True:
    time.sleep(60)

via tunnel: {'status': 'ok', 'model': 'strands-decider-2B-qwen3.5-v1-2610', 'checkpoint': 'StrandsAgents/strands-decider-2B-qwen3.5-v1-2610', 'base_model': 'Qwen/Qwen3.5-2B-Base', 'num_slots': 24, 'max_length': 4096, 'temperature': 1.0545210242263665, 'device': 'cuda', 'prefix_cache': True, 'vision': False}
serving. keep this cell running.


KeyboardInterrupt: 